
# RNN (Keras) para **Regresión** en *Boston Housing* — Guía Paso a Paso con Teoría

**Objetivo pedagógico:**  
Implementar una **Red Neuronal Recurrente (RNN)** con Keras/TensorFlow para un problema de **regresión** sobre el dataset **Boston Housing** de OpenML.  
Aunque el dataset es tabular (no secuencial), lo **reinterpretaremos como secuencia**: 13 características → **13 pasos de tiempo** (timesteps) con **1 feature por paso**. Esto se hace con fines **didácticos** para practicar RNN/LSTM/GRU.

> En la práctica, para datos tabulares suelen preferirse MLPs, Random Forest, XGBoost u otros modelos; aquí usamos RNN para aprender el flujo y la API.



## 1. Instalación y preparación del entorno
Ejecuta estas celdas si necesitas instalar las dependencias en tu entorno local (o en Colab).


In [ ]:

# (Opcional) Crear y activar entorno virtual en tu máquina local (no se ejecuta en notebook):
# python -m venv .venv
# source .venv/bin/activate  # Windows: .venv\Scripts\activate

# Instalar dependencias principales
# Nota: ejecuta estas líneas solo si te faltan las librerías.
# !pip install --upgrade pip
# !pip install tensorflow scikit-learn matplotlib pandas numpy



## 2. Imports con teoría
- **NumPy/Pandas** para manejo de datos.
- **Matplotlib** para visualización (sin estilos/colores personalizados).
- **scikit-learn** para split, escalado y métricas.
- **TensorFlow/Keras** para definir y entrenar la red.


In [ ]:

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.datasets import fetch_openml

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Mostrar versiones útiles
print("TensorFlow:", tf.__version__)



## 3. Semillas y **hiperparámetros** clave
- **Semillas** para reproducibilidad (hasta cierto punto).
- Hiperparámetros base: `EPOCHS`, `BATCH_SIZE`, tipo de RNN, unidades, tasa de aprendizaje.


In [ ]:

RANDOM_SEED = 42
TEST_SIZE = 0.2     # 20% para test
VAL_SIZE  = 0.2     # 20% de train para validación
EPOCHS = 120
BATCH_SIZE = 32
RNN_TYPE = "SimpleRNN"  # Cambia a "LSTM" o "GRU" para experimentar
RNN_UNITS = 32
LEARNING_RATE = 1e-3

np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)



## 4. Cargar datos: **Boston Housing** desde OpenML
- `load_boston` está **deprecado** en scikit-learn → usamos **OpenML**.
- La primera descarga requiere **internet**. Luego queda cacheado.


In [ ]:

try:
    boston = fetch_openml(name="boston", version=1, as_frame=True)
    X_df = boston.data
    y_series = boston.target.astype(float)
except Exception as e:
    print("⚠️ No fue posible descargar el dataset de OpenML.")
    print("Detalle:", e)
    print("Alternativa: usa California Housing (ej. fetch_california_housing) o carga un CSV local.")
    raise

print("Dimensiones:", X_df.shape)
print("Columnas:", list(X_df.columns))
X_df.head(3)  # Vista rápida



## 5. División **train/valid/test**
Separamos un conjunto **test** (para evaluación final) y reservamos una porción de **validación** desde el entrenamiento para monitorear el sobreajuste.


In [ ]:

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X_df.values, y_series.values, test_size=TEST_SIZE, random_state=RANDOM_SEED
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=VAL_SIZE, random_state=RANDOM_SEED
)

print("Tamaños -> train:", X_train.shape, "valid:", X_val.shape, "test:", X_test.shape)



## 6. **Escalamiento** de características
Estandarizamos (media 0, var 1) para estabilizar el entrenamiento de la red.


In [ ]:

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)



## 7. Reinterpretar **tabular → secuencia**
Una RNN espera tensores 3D: **(batch, timesteps, features)**.  
Aquí convertimos `(n, 13)` → `(n, 13, 1)` para tratar cada columna como un paso de tiempo.


In [ ]:

def to_sequence(arr_2d: np.ndarray) -> np.ndarray:
    return arr_2d.reshape((arr_2d.shape[0], arr_2d.shape[1], 1))

X_train_seq = to_sequence(X_train_scaled)
X_val_seq   = to_sequence(X_val_scaled)
X_test_seq  = to_sequence(X_test_scaled)

X_train_seq.shape



## 8. Definir el **modelo RNN** en Keras
Arquitectura base:
1. Capa recurrente (**SimpleRNN/LSTM/GRU**) para leer la secuencia.
2. Capa **Dense** intermedia (ReLU) para proyección no lineal.
3. Capa **Dense(1)** lineal para salida de **regresión**.


In [ ]:

def build_rnn_model(input_shape, rnn_type="SimpleRNN", units=32, lr=1e-3):
    model = keras.Sequential(name=f"{rnn_type}_regressor")
    if rnn_type == "LSTM":
        model.add(layers.LSTM(units, input_shape=input_shape))
    elif rnn_type == "GRU":
        model.add(layers.GRU(units, input_shape=input_shape))
    else:
        model.add(layers.SimpleRNN(units, input_shape=input_shape))
    model.add(layers.Dense(16, activation="relu"))
    model.add(layers.Dense(1, activation="linear"))
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),
                  loss="mse", metrics=["mae"])
    return model

model = build_rnn_model(
    input_shape=(X_train_seq.shape[1], X_train_seq.shape[2]),
    rnn_type=RNN_TYPE, units=RNN_UNITS, lr=LEARNING_RATE
)
model.summary()



## 9. **Entrenamiento**
Entrenamos por `EPOCHS` épocas y tamaño de batch `BATCH_SIZE`, monitoreando `val_loss`/`val_mae` para detectar sobreajuste.


In [ ]:

history = model.fit(
    X_train_seq, y_train,
    validation_data=(X_val_seq, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)



## 10. Visualización del rendimiento
Curvas de **pérdida (MSE)** y **MAE** para train/validación.


In [ ]:

# Pérdida (MSE)
plt.figure()
plt.plot(history.history["loss"], label="loss (train)")
plt.plot(history.history["val_loss"], label="loss (val)")
plt.title("Evolución de la pérdida (MSE)")
plt.xlabel("Época")
plt.ylabel("MSE")
plt.legend()
plt.tight_layout()

# MAE
plt.figure()
plt.plot(history.history["mae"], label="mae (train)")
plt.plot(history.history["val_mae"], label="mae (val)")
plt.title("Evolución del MAE")
plt.xlabel("Época")
plt.ylabel("MAE")
plt.legend()
plt.tight_layout()



## 11. Evaluación y **predicciones**
Calculamos **RMSE** y **MAE** sobre *test* y mostramos un **caso de prueba** (una fila real vs. predicción).


In [ ]:

y_pred = model.predict(X_test_seq).ravel()
rmse = mean_squared_error(y_test, y_pred, squared=False)
mae  = mean_absolute_error(y_test, y_pred)

print(f"RMSE (test): {rmse:.3f}")
print(f"MAE  (test): {mae:.3f}")

# Caso de prueba (primer ejemplo de test)
idx = 0
print("\n--- Caso de prueba ---")
print("Entrada original (primeras 5 features sin escalar):")
print(X_test[idx][:5])
print(f"Valor real (medv): {y_test[idx]:.2f}")
print(f"Predicción       : {y_pred[idx]:.2f}")



## 12. Guardar artefactos (opcional)
Exportamos las predicciones y, si lo deseas, puedes guardar también las figuras con `plt.savefig(...)`.


In [ ]:

results_df = pd.DataFrame({"y_true": y_test, "y_pred": y_pred})
results_path = "predicciones_test.csv"
results_df.to_csv(results_path, index=False)
results_df.head()



## 13. Experimentos sugeridos
- Cambiar `RNN_TYPE` a **"LSTM"** o **"GRU"** y re-entrenar.
- Probar **EarlyStopping** para detener el entrenamiento si `val_loss` deja de mejorar.
- Ajustar `RNN_UNITS`, `EPOCHS`, `BATCH_SIZE`, `LEARNING_RATE`.
- Comparar con un **MLP** (capas densas) para ver diferencias con datos tabulares.


In [ ]:

# EarlyStopping (opcional): vuelve a entrenar con paciencia de 10 épocas
# from tensorflow.keras.callbacks import EarlyStopping
# es = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
# history_es = model.fit(
#     X_train_seq, y_train,
#     validation_data=(X_val_seq, y_val),
#     epochs=EPOCHS,
#     batch_size=BATCH_SIZE,
#     callbacks=[es],
#     verbose=1
# )
# plt.figure()
# plt.plot(history_es.history["val_loss"])
# plt.title("val_loss con EarlyStopping")
# plt.xlabel("Época")
# plt.ylabel("MSE")
# plt.tight_layout()



## 14. Conclusiones
- Viste el flujo completo para **RNN en regresión** con datos tabulares (reinterpretados como secuencia).
- Las curvas de pérdida/MAE permiten inspeccionar **underfitting**/**overfitting**.
- La comparación con **LSTM/GRU** puede revelar diferencias sutiles de capacidad.
- En problemas **tabulares reales**, prueba también **MLP, árboles y boosting** para establecer una línea base fuerte.
